# Model Updater

Keeps the delay model current without interrupting the predictor.

Daily flow: fetch new SBB Ist Daten → check retrain triggers → train into inactive slot → flip active pointer → sleep 24 h.

---

### Data — default.istdaten
The shared cluster table (iceberg.sbb.istdaten) is read-only. init() copies it into default.istdaten, a personal Iceberg table in the user's Spark warehouse (hdfs:/user/{username}/final/warehouse/). New days fetched from opentransportdata.swiss are appended here daily.

### Retrain triggers (either one fires a retrain)
| Trigger | Logic |
|---|---|
| Time-based | ≥ RETRAIN_INTERVAL_DAYS days since the training-end date in model_trained_dates |
| Divergence | q=0.5 pinball loss on recent data deviates from the post-train baseline by more than the threshold |

### Blue-green slots
Training always writes into the inactive slot. On success the active_model pointer is flipped  the predictor is never blocked.

```
artifacts/model/
    model_a/             ← xgb model, hist aggs, encoders …
    model_b/             ← same structure (the other slot)
    active_model         ← "model_a" or "model_b"
    istdaten_dates       ← START= / END= of rows in default.istdaten
    model_trained_dates  ← START= / END= of the last training window
```

In [1]:
import time
import traceback
from datetime import datetime, timedelta

from src.models.model_updator import ModelUpdator
from src.config.settings import get_settings
from src.config.spark_session import get_spark_session

In [2]:
settings = get_settings(
    # istdaten_table="iceberg.sbb.istdaten"
)
spark = get_spark_session(settings)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
# # Quick test for creating table permission:

# spark.sql("CREATE TABLE iceberg.sbb.test_write (id INT) USING iceberg")

In [3]:
# Parameters
RUN_INTERVAL_HOURS  = 24          # hours between loops (24 = daily)
RETRAIN_INTERVAL_DAYS = 7         # self update frequency (7 = weekly)
REGION_UUIDS        = None
FORCE_RETRAIN       = False
MAX_ITERATIONS      = None        # None = run forever

## One-time initialisation

Run once before starting the loop for the first time. It:
1. Copies iceberg.sbb.istdaten → default.istdaten in your personal Spark warehouse (2025-01-01 onward).
2. Writes istdaten_dates and clears failed_istdaten_days.

Safe to re-run, skips the copy if the table already exists. Set force_copy=True to drop and recreate.

In [4]:
updator = ModelUpdator(spark=spark, retrain_interval_days=RETRAIN_INTERVAL_DAYS)
updator.init(force_copy=True)

=== ModelUpdator.init() ===
  copying iceberg.sbb.istdaten -> default.istdaten via Spark (from 2025-01-01 onwards) ...


  done.
  istdaten_dates written: START=2025-01-01  END=2026-01-31


In [5]:
table = settings.spark_user_istdaten_table
print(f"Trying to read via Spark: {table}")
spark.table(table).limit(5).show()

Trying to read via Spark: default.istdaten


+-------------+--------------------+-----------+-------------+-----------------+----------+-------+---------+----------+---------+---------+------+-------+------------------+-------------------+-------------------+----------+-------------------+-------------------+----------+-------+
|operating_day|             trip_id|operator_id|operator_abrv|    operator_name|product_id|line_id|line_text|circuit_id|transport|unplanned|failed|  bpuic|         stop_name|           arr_time|         arr_actual|arr_status|           dep_time|         dep_actual|dep_status|transit|
+-------------+--------------------+-----------+-------------+-----------------+----------+-------+---------+----------+---------+---------+------+-------+------------------+-------------------+-------------------+----------+-------------------+-------------------+----------+-------+
|   2025-06-12|ch:1:sjyid:100073...|      85:96|      AVA-wsb|Aargau Verkehr AG|       Zug|   4242|      S14|          |        S|    false| fals

In [6]:
spark.sql("SHOW TABLES IN default").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [12]:
### In case can't create new namespace, just use the same table for now:

# # Point everything at the shared table Spark can already see
# from dataclasses import replace
# updator.settings = replace(updator.settings, 
#     user_schema="iceberg.sbb",
#     istdaten_table="iceberg.sbb.istdaten"
# )

# # Manually write the date range so run() knows what data is available
# from src.models.model_updator import write_dates
# write_dates("istdaten_dates", start="2025-01-01", end="2026-01-31", settings=updator.settings)
# write_dates("model_trained_dates", start="2025-01-01", end="2026-01-31", settings=updator.settings)

## Daily loop

Each iteration:
- fetches any new istdaten days up to yesterday,
- retrains the model if ≥ `RETRAIN_INTERVAL_DAYS` have passed since the last training end date,
- sleeps until the next 24-hour mark.

Interrupt the kernel to stop cleanly.

In [7]:
def _log(msg: str) -> None:
    """Print with a timestamp prefix."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")


iteration = 0
_log("Daily update loop starting.")

while True:
    iteration += 1
    _log(f"─── Iteration {iteration} ───────────────────────────────────────")

    # Re-use the same updator (keeps Spark session and Trino connection alive)
    try:
        result = updator.run(
            region_uuids=REGION_UUIDS,
            force_retrain=FORCE_RETRAIN,
        )
        FORCE_RETRAIN = False  # reset one-shot flag after first use

        _log(f"New days appended : {result['new_days']}")
        if result["retrained"]:
            _log(f"Model retrained   : slot={result['slot']}")
            _log(f"Train result      : {result.get('train_result')}")
        else:
            _log("No retrain this cycle.")

    except KeyboardInterrupt:
        _log("Interrupted by user — exiting loop.")
        break
    except Exception:
        _log("ERROR during run():")
        traceback.print_exc()
        _log("Continuing to next iteration despite error.")

    if MAX_ITERATIONS is not None and iteration >= MAX_ITERATIONS:
        _log(f"Reached MAX_ITERATIONS={MAX_ITERATIONS} — stopping.")
        break

    next_run = datetime.now() + timedelta(hours=RUN_INTERVAL_HOURS)
    _log(f"Sleeping {RUN_INTERVAL_HOURS}h — next run at {next_run.strftime('%Y-%m-%d %H:%M:%S')}")
    try:
        time.sleep(RUN_INTERVAL_HOURS * 3600)
    except KeyboardInterrupt:
        _log("Interrupted during sleep — exiting loop.")
        break

_log("Loop finished.")

[2026-05-29 09:48:32] Daily update loop starting.
[2026-05-29 09:48:32] ─── Iteration 1 ───────────────────────────────────────
=== ModelUpdator.run() ===
  2026-02-01 not on website yet, stopping.
  Appended 0 new day(s).
  Retrain triggered — reason: ≥7 days since last training
  Retraining into slot: model_b


ERROR:root:KeyboardInterrupt while sending command.         (0 + 16) / 106][Stage 9:>                                                   (0 + 0) / 106]
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/com-490/tljh/user/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


[2026-05-29 09:48:47] Interrupted by user — exiting loop.
[2026-05-29 09:48:47] Loop finished.


Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out                                                               
	at java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:409)
	at java.net.ServerSocket.implAccept(ServerSocket.java:560)
	at java.net.ServerSocket.accept(ServerSocket.java:528)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)
